# Phase 1 Colab runner: where does forgetting live?

All logic lives in `src/`. Runtime: GPU (T4).

Every step is separate and ends with a push, so a disconnect loses at most one step. In a new session, rerun **Setup** first, then continue where you stopped.

Order: setup, fine-tune seed 1, push, fine-tune seed 2, push, components for seed 0, 1, 2 (push after each), table, push.

## Setup (every session)

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

REPO = 'sohilkhan-2611/-clip-zero-shot-forgetting'
BRANCH = 'phase-1'
CKPT_DIR = '/content/drive/MyDrive/clip-forgetting/checkpoints'
DATA_DIR = '/content/data'   # local disk, re-downloaded each session
token = userdata.get('GITHB-TOKEN')

In [ ]:
import os
if not os.path.exists('/content/repo'):
    !git clone -q -b {BRANCH} https://{token}@github.com/{REPO}.git /content/repo
%cd /content/repo
!git pull -q
!pip install -q -e .[dev]
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la {CKPT_DIR}

## 1. Fine-tune seeds 1 and 2
Seed 0 already exists from Phase 0 (check it is listed above). Each run takes roughly 35 minutes (Phase 0 timing) and writes its checkpoint to Drive.

In [ ]:
!python -m src.finetune --config configs/finetune_flowers_vitb16.yaml --set seed=1 data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR} output.results_dir=results/phase1

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/finetune_*.json
!git commit -q -m 'Phase 1: Flowers102 fine-tune, seed 1' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.finetune --config configs/finetune_flowers_vitb16.yaml --set seed=2 data.root={DATA_DIR} output.checkpoint_dir={CKPT_DIR} output.results_dir=results/phase1

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/finetune_*.json
!git commit -q -m 'Phase 1: Flowers102 fine-tune, seed 2' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

## 2. Revert / retain per component
One checkpoint per cell: 10 conditions (pre, ft, 4 revert, 4 retain), each evaluated on Flowers (2,000 images) and the five held-out sets. Results are saved after every condition; if a cell stops, just run it again and it continues where it stopped.

For seed 0, the `ft` numbers should equal the Phase 0 fine-tune result exactly. That is a free consistency check.

In [ ]:
!python -m src.components --config configs/phase1_components_flowers.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_flowers_vitb16_seed0.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: components, seed 0' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.components --config configs/phase1_components_flowers.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_flowers_vitb16_seed1.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: components, seed 1' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

In [ ]:
!python -m src.components --config configs/phase1_components_flowers.yaml --set data.root={DATA_DIR} checkpoints=[{CKPT_DIR}/finetune_flowers_vitb16_seed2.pt]

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/components_*.json
!git commit -q -m 'Phase 1: components, seed 2' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1

## 3. Table over seeds

In [ ]:
!python -m src.aggregate_phase1 --inputs 'results/phase1/components_flowers_*.json' --out results/phase1/summary_components_flowers.json

In [ ]:
!git config user.name 'colab-runner'
!git config user.email 'colab-runner@users.noreply.github.com'
!git add results/phase1/summary_*.json
!git commit -q -m 'Phase 1: summary table over seeds' || echo 'nothing to commit'
!git pull -q --rebase origin {BRANCH}
!git push -q origin {BRANCH}
!git status -sb | head -1
!git log --oneline -1